In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; free -g | head -2; python3 --version; df -h /content | tail -1

In [ ]:
%%bash
cd /content
if [ ! -d new ]; then git clone -q -b perf/long-takes-speed-tiers https://github.com/adamholter/folio new; else (cd new && git pull -q); fi
[ -d base ] || git clone -q https://github.com/campeterson41/folio base
cd new && git log --oneline | head -3
python3 tools/setup_studio_runtime.py 2>&1 | tail -3
echo SETUP_DONE

In [ ]:
%%bash
cd /content
which ffmpeg || (apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y ffmpeg >/dev/null 2>&1)
which ffmpeg; ffmpeg -version | head -1
curl -sL -o moby.mp3 https://archive.org/download/moby_dick_librivox/mobydick_001_002_melville.mp3
ls -la moby.mp3
ffmpeg -nostdin -v error -y -i moby.mp3 -ac 1 -ar 48000 -c:a pcm_s24le moby_full.wav
ffmpeg -nostdin -v error -y -ss 60 -t 60 -i moby_full.wav -c:a pcm_s24le moby_60s.wav
ffmpeg -nostdin -v error -y -ss 60 -t 600 -i moby_full.wav -c:a pcm_s24le moby_600s.wav
ls -la *.wav; sha256sum moby_60s.wav; echo AUDIO_DONE

In [ ]:
%%bash
cd /content
python3 - <<'PY'
import subprocess,sys,os
env=dict(os.environ)
py=os.path.expanduser('~/.local/share/folio/f-venv/bin/python')
r=subprocess.run([py,'new/tools/benchmark_studio.py','--input','moby_60s.wav','--workdir','bench60','--device','cuda','--baseline','base'],capture_output=True,text=True)
print(r.stdout[-6000:]);print(r.stderr[-3000:])
PY
echo BENCH60_BEGIN; cat bench60/benchmark.json; echo BENCH60_END

In [ ]:
%%bash
cd /content
python3 - <<'PY'
import subprocess,sys,os
py=os.path.expanduser('~/.local/share/folio/f-venv/bin/python')
r=subprocess.run([py,'new/tools/benchmark_studio.py','--input','moby_600s.wav','--workdir','bench600','--device','cuda','--baseline','base'],capture_output=True,text=True)
print(r.stdout[-6000:]);print(r.stderr[-3000:])
PY
echo BENCH600_BEGIN; cat bench600/benchmark.json; echo BENCH600_END
python3 - <<'PY'
import json
for n in ('bench60','bench600'):
    for run in ('baseline','reference','high','balanced','fast','draft'):
        try:
            d=json.load(open(f'/content/{n}/{run}.json'));p=d['native_restoration']
            print('RECEIPT',n,run,json.dumps(dict(model_seconds=p['runtime_seconds'],window_median=p.get('window_seconds_median'),gpu_peak_mib=p.get('gpu_peak_mib'),params=p['parameters'],tmp=d.get('temporary_peak_bytes'),quality=d.get('quality'))))
        except Exception as e:print('RECEIPT',n,run,'missing',e)
PY